# CodeMitra on a Kaggle GPU

Runs Ollama on this notebook's GPU so CodeMitra on your laptop can use a bigger GPU at full context (CodeMitra D-060).

**Easiest:** you do not need to open this at all: `codemitra kaggle start` on your laptop uploads and runs it for you.

**By hand:** Settings (right panel) → *Accelerator* **GPU T4 x2** · *Internet* **On** (needs a verified phone).
Then **Run All** and keep this tab open. The tunnel cell prints one line to paste on your laptop:

    codemitra remote set https://<name>.trycloudflare.com/<token> --ctx 262144

**When you are done:** stop the session (power button, top right) to save your weekly GPU hours, and on the laptop
run `codemitra remote off`. A session ends by itself after 12 hours; the URL changes every session.

The token is new every session and is part of the URL: anyone without it gets 404. Your code and prompts pass through
this Kaggle machine and Cloudflare's tunnel.

In [ ]:
# Settings: the model and the context CodeMitra will use (qwen3.5's maximum is 256K)
MODEL = "qwen3.5:latest"
CONTEXT = 262144
PORT = 8000
IDLE_MIN = 0  # by hand: run until you stop the session

import secrets
TOKEN = secrets.token_urlsafe(24)

def say(stage, **kw):
    # one line CodeMitra reads from the live log: MITRA|stage|key=value (no quotes, pipes or backslashes in values)
    clean = lambda v: str(v).replace("|", "/").replace('"', "'").replace("\\", "/").replace("\n", " ")
    print("MITRA|" + stage + "".join(f"|{k}={clean(v)}" for k, v in kw.items()), flush=True)

say("started", model=MODEL, context=CONTEXT)
print("model", MODEL, "| context", CONTEXT)

In [ ]:
# Install Ollama (about 1 minute) and show the GPUs
say("installing")
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# Start Ollama: flash attention + q8_0 KV cache (half the context memory), one model kept loaded
import json, os, subprocess, time, urllib.request

def api(path, body=None, timeout=30):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request("http://127.0.0.1:11434" + path, data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode() or "{}")

env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(CONTEXT),
           OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="-1",
           OLLAMA_NUM_PARALLEL="1", OLLAMA_MAX_LOADED_MODELS="1")
ollama_log = open("/kaggle/working/ollama.log", "w")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Ollama", api("/api/version")["version"], "is up")
        break
    except Exception:
        time.sleep(1)
else:
    say("error", detail="Ollama did not start")
    raise SystemExit("Ollama did not start: see /kaggle/working/ollama.log")

In [ ]:
# Download the model (a few minutes) and load it once at the full context, so CodeMitra's first call is fast
say("pulling", model=MODEL)
!ollama pull {MODEL}
say("loading", model=MODEL, context=CONTEXT)
t0 = time.time()
api("/api/generate", {"model": MODEL, "prompt": "Say ok.", "stream": False,
                      "options": {"num_ctx": CONTEXT}, "keep_alive": -1}, timeout=1800)
for m in api("/api/ps").get("models", []):
    size, vram = m.get("size", 0), m.get("size_vram", 0)
    where = "fully on GPU" if vram >= size else f"{100 * vram / max(size, 1):.0f}% on GPU (the rest on CPU: slower)"
    print(f"{m['name']}: {size / 1e9:.1f} GB at {m.get('context_length')} context, {where}, "
          f"loaded in {time.time() - t0:.0f} s")
    say("loaded", model=m["name"], gb=round(size / 1e9, 1), context=m.get("context_length"), where=where,
        seconds=round(time.time() - t0))
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
%%writefile /kaggle/working/proxy.py
"""Token proxy in front of Ollama, for the Kaggle notebook (D-060). Standard library only.

Ollama has no login, and the Cloudflare quick tunnel makes it public. This proxy answers only requests whose path
starts with /<token>/ (e.g. /<token>/api/chat), strips the token and forwards the rest to Ollama, streaming the reply
back as it comes (chat replies arrive token by token). Anything else gets 404, so a found URL without the token is
useless.

    python proxy.py <token> [port=8000] [ollama=127.0.0.1:11434] [state_dir]

With a state_dir (the notebook passes /kaggle/working): `POST /<token>/mitra/stop` writes `stop` there (the notebook
then shuts everything down, which ends the Kaggle run), and every model request touches `last_request`, so the
notebook can stop by itself when CodeMitra has not used it for a while.
"""
from __future__ import annotations

import hmac
import http.client
import os
import sys
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

QUIET = {"/api/tags", "/api/ps", "/api/version", "/api/show"}  # status checks: not "CodeMitra is using it"

HOP = {"connection", "keep-alive", "transfer-encoding", "te", "trailer", "upgrade", "proxy-authorization",
       "proxy-authenticate", "host", "content-length"}


def _touch(path: str) -> None:
    with open(path, "a", encoding="utf-8"):
        pass
    os.utime(path, None)


def make_handler(token: str, upstream: str, state_dir: str | None = None):
    host, _, port = upstream.partition(":")
    port_n = int(port or 11434)

    class Handler(BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"

        def log_message(self, fmt, *args):  # keep the notebook quiet; the token must not appear in logs
            pass

        def _refuse(self) -> None:
            self.send_response(404)
            self.send_header("Content-Length", "0")
            self.end_headers()

        def _forward(self) -> None:
            length = int(self.headers.get("Content-Length") or 0)
            body = self.rfile.read(length) if length else None  # read first: a refused client gets a clean 404
            parts = self.path.split("/", 2)  # "", token, rest
            if len(parts) < 2 or not hmac.compare_digest(parts[1].encode(), token.encode()):
                self._refuse()
                return
            tail = "/" + (parts[2] if len(parts) > 2 else "")
            if tail.startswith("/mitra/"):  # CodeMitra's own commands, never forwarded to Ollama
                ok = tail == "/mitra/ping" or (tail == "/mitra/stop" and state_dir is not None)
                if ok and tail == "/mitra/stop":
                    _touch(os.path.join(state_dir, "stop"))
                msg = (b'{"ok": true, "time": %d}' % int(time.time())) if ok else b'{"ok": false}'
                self.send_response(200 if ok else 404)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(msg)))
                self.end_headers()
                self.wfile.write(msg)
                return
            if state_dir is not None and tail.split("?")[0] not in QUIET:
                _touch(os.path.join(state_dir, "last_request"))
            headers = {k: v for k, v in self.headers.items() if k.lower() not in HOP}
            conn = http.client.HTTPConnection(host, port_n, timeout=3600)  # a long generation is normal
            try:
                conn.request(self.command, tail, body=body, headers=headers)
                resp = conn.getresponse()
                self.send_response(resp.status)
                for k, v in resp.getheaders():
                    if k.lower() not in HOP:
                        self.send_header(k, v)
                self.send_header("Transfer-Encoding", "chunked")
                self.end_headers()
                while True:
                    chunk = resp.read1(65536)
                    if not chunk:
                        break
                    self.wfile.write(b"%x\r\n%s\r\n" % (len(chunk), chunk))
                    self.wfile.flush()
                self.wfile.write(b"0\r\n\r\n")
                self.wfile.flush()
            except (ConnectionError, OSError) as exc:
                try:
                    msg = f'{{"error": "proxy: {type(exc).__name__}"}}'.encode()
                    self.send_response(502)
                    self.send_header("Content-Type", "application/json")
                    self.send_header("Content-Length", str(len(msg)))
                    self.end_headers()
                    self.wfile.write(msg)
                except OSError:
                    pass
            finally:
                conn.close()

        do_GET = do_POST = do_DELETE = do_PUT = _forward

    return Handler


def serve(token: str, port: int = 8000, upstream: str = "127.0.0.1:11434",
          state_dir: str | None = None) -> ThreadingHTTPServer:
    server = ThreadingHTTPServer(("127.0.0.1", port), make_handler(token, upstream, state_dir))
    server.daemon_threads = True
    return server


if __name__ == "__main__":
    tok = sys.argv[1]
    prt = int(sys.argv[2]) if len(sys.argv) > 2 else 8000
    up = sys.argv[3] if len(sys.argv) > 3 else "127.0.0.1:11434"
    state = sys.argv[4] if len(sys.argv) > 4 else None
    print(f"proxy on 127.0.0.1:{prt} -> {up}", flush=True)
    serve(tok, prt, up, state).serve_forever()


In [ ]:
# Start the token proxy and a Cloudflare quick tunnel (no account needed), then print the line for your laptop
import re, sys, threading
say("tunnel")
proxy = subprocess.Popen([sys.executable, "/kaggle/working/proxy.py", TOKEN, str(PORT), "127.0.0.1:11434",
                          "/kaggle/working"],
                         stdout=open("/kaggle/working/proxy.log", "w"), stderr=subprocess.STDOUT)
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /kaggle/working/cloudflared
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                           "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public = None
tunnel_log = open("/kaggle/working/tunnel.log", "w")
t_end = time.time() + 90
while time.time() < t_end and public is None:
    line = tunnel.stdout.readline()
    tunnel_log.write(line)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        public = found.group(0)
if not public:
    say("error", detail="no tunnel URL")
    raise SystemExit("No tunnel URL: see /kaggle/working/tunnel.log")
threading.Thread(target=lambda: [tunnel_log.write(l) for l in tunnel.stdout], daemon=True).start()
URL = f"{public}/{TOKEN}"
for _ in range(30):  # a new tunnel name takes a few seconds to resolve
    try:
        with urllib.request.urlopen(URL + "/api/tags", timeout=15) as r:
            names = [m["name"] for m in json.loads(r.read().decode()).get("models", [])]
        print("✓ the tunnel works · models:", ", ".join(names))
        break
    except Exception:
        time.sleep(3)
else:
    print("⚠ the tunnel did not answer yet; try the line below anyway in a minute")
say("ready", url=public, model=MODEL, context=CONTEXT)  # the token is not printed: CodeMitra already has it
print()
print("On your laptop, run:")
print()
print(f"    codemitra remote set {URL} --ctx {CONTEXT}")
print()
print("then start CodeMitra. Keep this tab open.")

In [ ]:
# Keep the session alive and show its state every 5 minutes. Stops when CodeMitra says `codemitra kaggle stop`, or after
# IDLE_MIN minutes without a model request (0 = never), and then shuts everything down: the Kaggle run ends.
started = last_seen = time.time()
reason, next_report = "", 0.0
while not reason:
    if os.path.exists("/kaggle/working/stop"):
        reason = "codemitra kaggle stop"
    elif os.path.exists("/kaggle/working/last_request"):
        last_seen = max(last_seen, os.path.getmtime("/kaggle/working/last_request"))
    if not reason and IDLE_MIN and time.time() - last_seen > IDLE_MIN * 60:
        reason = f"idle {IDLE_MIN} min"
    if not reason and (ollama.poll() is not None or tunnel.poll() is not None):
        reason = "ollama or the tunnel stopped"
    if time.time() >= next_report:
        try:
            ps = api("/api/ps").get("models", [])
            loaded = ", ".join(f"{m['name']} {m.get('size_vram', 0) / 1e9:.1f} GB on GPU" for m in ps) or "none loaded"
        except Exception as exc:
            loaded = f"Ollama not answering: {exc}"
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip().replace("\n", " | ")
        idle = int((time.time() - last_seen) / 60)
        print(time.strftime("%H:%M"), "·", loaded, "· GPU", gpu, f"· idle {idle} min", flush=True)
        say("alive", idle_min=idle, up_min=int((time.time() - started) / 60))
        next_report = time.time() + 300
    time.sleep(15)
say("stopping", reason=reason, up_min=int((time.time() - started) / 60))
for p in (tunnel, proxy, ollama):
    p.terminate()
print("stopped:", reason)